# 10. 傾向分析 — 練習問題

**対象技術**: 量子コンピューティング（超伝導量子プロセッサの量子ビット数推移）

傾向分析は、過去の時系列データに曲線をフィットし、将来へ外挿して閾値到達時期を推定する手法である。本ノートブックでは、超伝導方式の量子ビット数の年次データに指数モデルとロジスティックモデルをフィットし、両モデルで CRQC 相当の閾値到達年を外挿する。残差ブートストラップで到達年の予測区間も求める。

必要なライブラリを読み込む。

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt

# --- 日本語フォント設定: 共通モジュール jp_font.py を読み込む ---
# フォント探索・登録・フォールバックの実装は repo 直下の jp_font.py に集約。
import os as _os, sys as _sys
_d = _os.path.abspath(_os.getcwd())
while not _os.path.exists(_os.path.join(_d, "jp_font.py")) and _d != _os.path.dirname(_d):
    _d = _os.path.dirname(_d)
_sys.path.insert(0, _d)
from jp_font import setup_japanese_font
setup_japanese_font()


超伝導量子プロセッサの量子ビット数の年次データと、CRQC 相当の参照閾値を定義する。

In [ ]:
# 超伝導量子プロセッサの量子ビット数の年次推移（公表値に基づく代表データ）
YEARS = np.array([2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024],
                 dtype=float)
QUBITS = np.array([5, 17, 72, 53, 65, 127, 433, 1121, 1180], dtype=float)

# CRQC相当に向けた物理量子ビット規模の参照閾値。
# 誤り訂正のオーバーヘッドを織り込むと、RSA-2048解読には
# 100万量子ビット級の物理量子ビットが必要との見積りが代表的。
THRESHOLD = 1_000_000.0

指数モデル q=a*exp(b*t) を対数線形回帰でフィットする関数を定義する。

In [ ]:
def fit_exponential(t, y):
    """指数モデル q=a*exp(b*t) を対数線形回帰でフィット。(a,b,t0) を返す。"""
    t0 = t[0]                       # 年を基準年からの経過年に変換し数値安定化
    x = t - t0
    logy = np.log(y)
    # log y = log a + b*x の最小二乗（1次多項式フィット）
    b, log_a = np.polyfit(x, logy, 1)
    return np.exp(log_a), b, t0


def exp_predict(a, b, t0, t):
    return a * np.exp(b * (t - t0))

ロジスティックモデル q=L/(1+exp(-b*(t-t0))) をグリッド探索でフィットする関数を定義する。

In [ ]:
def fit_logistic(t, y):
    """ロジスティックモデルをグリッド探索でフィット。(L,b,t0_mid) を返す。

    L・b・tm の3軸グリッドを numpy のブロードキャストで一括評価し、
    SSE 最小の組を選ぶ（明示ループより高速で、ブートストラップ反復に耐える）。
    """
    t0 = t[0]
    x = (t - t0)[:, None, None, None]            # データ軸
    yv = y[:, None, None, None]
    # 飽和水準 L・成長率 b・変曲点 tm の3軸グリッド
    L = np.linspace(y.max() * 1.5, y.max() * 30, 40)[None, :, None, None]
    b = np.linspace(0.2, 2.0, 30)[None, None, :, None]
    tm = np.linspace(x.min() - 2, x.max() + 12, 35)[None, None, None, :]
    pred = L / (1.0 + np.exp(-b * (x - tm)))
    sse = np.sum((pred - yv) ** 2, axis=0)       # データ軸で和 → (L,b,tm)
    i, j, k = np.unravel_index(np.argmin(sse), sse.shape)
    return L[0, i, 0, 0], b[0, 0, j, 0], tm[0, 0, 0, k], t0


def logistic_predict(L, b, tm, t0, t):
    return L / (1.0 + np.exp(-b * ((t - t0) - tm)))

各モデルで参照閾値に到達する年を求める関数を定義する。

In [ ]:
def reach_year_exp(a, b, t0, threshold, search_max=2080):
    """指数モデルで閾値に達する年を求める（解析解）。"""
    if a <= 0 or b <= 0:
        return None
    yr = t0 + np.log(threshold / a) / b
    return yr if yr <= search_max else None


def reach_year_logistic(L, b, tm, t0, threshold, search_max=2080):
    """ロジスティックモデルで閾値に達する年を求める。"""
    if L <= threshold:
        return None                 # 飽和水準が閾値以下 → 永遠に到達しない
    # 走査で初めて閾値を超える年を探す
    for yr in np.arange(YEARS[-1], search_max, 0.1):
        if logistic_predict(L, b, tm, t0, yr) >= threshold:
            return yr
    return None

残差ブートストラップで到達年の分布（予測区間）を求める関数を定義する。

In [ ]:
def bootstrap_reach_year(t, y, fit_fn, reach_fn, n_boot, rng):
    """残差ブートストラップで到達年の分布を求める。"""
    # 元データでフィットし残差を得る
    if fit_fn == "exp":
        a, b, t0 = fit_exponential(t, y)
        base_pred = exp_predict(a, b, t0, t)
    else:
        L, b, tm, t0 = fit_logistic(t, y)
        base_pred = logistic_predict(L, b, tm, t0, t)
    resid = y - base_pred

    reach_years = []
    for _ in range(n_boot):
        # 残差を復元抽出してデータに足し戻し再フィット
        y_b = np.clip(base_pred + rng.choice(resid, size=len(y),
                                             replace=True), 1.0, None)
        if fit_fn == "exp":
            a, b, t0 = fit_exponential(t, y_b)
            yr = reach_year_exp(a, b, t0, THRESHOLD)
        else:
            L, b, tm, t0 = fit_logistic(t, y_b)
            yr = reach_year_logistic(L, b, tm, t0, THRESHOLD)
        if yr is not None:
            reach_years.append(yr)
    return np.array(reach_years)

両モデルをフィットし、閾値到達年（点推定）を計算・表示する。

In [ ]:
rng = np.random.default_rng(11)

# --- 指数モデル ---
a, b, t0 = fit_exponential(YEARS, QUBITS)
exp_reach = reach_year_exp(a, b, t0, THRESHOLD)
# --- ロジスティックモデル ---
L, lb, tm, lt0 = fit_logistic(YEARS, QUBITS)
log_reach = reach_year_logistic(L, lb, tm, lt0, THRESHOLD)

print("=" * 64)
print("傾向分析 : 超伝導量子ビット数の推移と閾値到達年の外挿")
print("=" * 64)
print(f"参照閾値 : {THRESHOLD:.0f} 量子ビット")
print(f"\n[指数モデル]      q = {a:.2f} * exp({b:.3f}*(t-{t0:.0f}))")
print(f"  年成長率 約 {(np.exp(b)-1)*100:.0f}% / 閾値到達年(点推定) "
      f"{exp_reach:.1f}" if exp_reach else "  閾値到達せず")
print(f"\n[ロジスティックモデル] 飽和水準 L = {L:.0f} 量子ビット")
if log_reach:
    print(f"  閾値到達年(点推定) {log_reach:.1f}")
else:
    print(f"  飽和水準が閾値未満 → このモデルでは閾値に到達しない")

残差ブートストラップで両モデルの到達年の予測区間を求める。

In [ ]:
print("--- 残差ブートストラップによる到達年の予測区間（2000試行）---")
exp_boot = bootstrap_reach_year(YEARS, QUBITS, "exp",
                                reach_year_exp, 2000, rng)
if len(exp_boot) > 0:
    print(f"  指数モデル        : 中央値 {np.median(exp_boot):.0f}年, "
          f"90%予測区間 {np.percentile(exp_boot,5):.0f}"
          f"-{np.percentile(exp_boot,95):.0f}年")
log_boot = bootstrap_reach_year(YEARS, QUBITS, "log",
                                reach_year_logistic, 300, rng)
if len(log_boot) > 0:
    rate = len(log_boot) / 300 * 100
    print(f"  ロジスティックモデル: 中央値 {np.median(log_boot):.0f}年, "
          f"90%予測区間 {np.percentile(log_boot,5):.0f}"
          f"-{np.percentile(log_boot,95):.0f}年")
    print(f"      （ブートストラップ標本のうち到達したのは {rate:.0f}%）")
else:
    print("  ロジスティックモデル: 多くの標本で閾値に到達しない")

## 可視化: データ点と両モデルのフィット曲線

観測された量子ビット数を散布図で示し、指数フィット曲線とロジスティックフィット曲線を重ね描く。参照閾値を水平線で引き、各モデルの到達年を縦軸対数で注記する。指数モデルとロジスティックモデルで到達年が大きく食い違うことが視覚的に確認できる。

In [ ]:
t_plot = np.linspace(YEARS[0], 2045, 300)
fig, ax = plt.subplots(figsize=(9, 6))
ax.scatter(YEARS, QUBITS, color="black", zorder=5, label="Observed")
ax.plot(t_plot, exp_predict(a, b, t0, t_plot), "--", color="crimson",
        label="Exponential fit")
ax.plot(t_plot, logistic_predict(L, lb, tm, lt0, t_plot), "-",
        color="steelblue", label="Logistic fit")
ax.axhline(THRESHOLD, color="gray", ls=":", label="Threshold (1e6 qubits)")

# 到達年の注記
if exp_reach:
    ax.axvline(exp_reach, color="crimson", ls=":", alpha=0.6)
    ax.annotate(f"exp reach\n{exp_reach:.0f}", (exp_reach, THRESHOLD),
                color="crimson", fontsize=8, ha="left", va="bottom")
if log_reach:
    ax.axvline(log_reach, color="steelblue", ls=":", alpha=0.6)
    ax.annotate(f"logistic reach\n{log_reach:.0f}", (log_reach, THRESHOLD),
                color="steelblue", fontsize=8, ha="left", va="bottom")
else:
    ax.annotate("logistic: saturates below threshold", (2030, THRESHOLD),
                color="steelblue", fontsize=8, ha="center", va="bottom")

ax.set_yscale("log")
ax.set_xlabel("Year")
ax.set_ylabel("Qubit count (log scale)")
ax.set_title("Superconducting qubit count: trend extrapolation")
ax.legend()
ax.grid(alpha=0.3, which="both")
plt.show()

print("[解釈] 指数モデルとロジスティックモデルで到達年は大きく食い違う。")
print("       この食い違い自体が発見であり、物理量子ビットの指数増加と")
print("       論理量子ビットへの変換効率は別問題だと明示する必要がある。")

## 未来デザイン論文での使われ方と結論への影響

未来デザインの論文において傾向分析は、過去の時系列データに数学的なカーブをフィットし、その曲線を将来へ延長して予測値や閾値到達年を求めるために用いられる。論文の典型的な使われ方は、指標の伸びを定量化し「この性能はX年に閾値に達する」と予測区間つきで提示することである。したがってこの手法が生み出す結論は、外挿された予測値という型をとり、数値と年限という最も明快で具体的な形で提示される。

結論の型は単純明快だが、その明快さは強い連続性バイアスと引き換えに得られている。時間観の面でこの手法は、未来を過去の延長として扱う立場を最も純粋に体現しており、過去のトレンドを生んだ力学が将来も変わらず働き続けると仮定する。境界設定の面では、フィットに用いたデータの範囲と選んだ関数形が外挿の形を決め、それ以外の情報——制度変化や新規参入——は結論に入り込まない。価値の所在は、どの曲線（指数かロジスティックか）を選ぶかという段階に埋め込まれ、関数形の選択がそのまま将来像の選択になる。

最大の限界は、技術的な不連続性を構造的に過小評価することである。飽和、断絶、新パラダイムへの乗り換えといった出来事は過去データに前例がなく、カーブのあてはめでは表現できないため、結論からこぼれ落ちる。結果として、この手法に依拠した論文の結論は、本稿で扱う諸手法のなかで最も明快な「数値＋年」を提示できる一方、その数値は最も脆い仮定——未来は過去の延長である——の上に立っている。予測値の鋭さと仮定の脆さは表裏一体であり、両方をあわせて読む必要がある。

## 発展課題

**課題A**: 両モデルの当てはまりを比較する指標（決定係数 R^2、AIC など）を計算し、どちらのモデルがデータをよく説明するかを判定せよ。

**課題B**: 新しい年のデータ点を1〜2個 YEARS / QUBITS に追加し、指数モデルの到達年がどれだけ動くかを観察せよ。短期データへの過剰適合と、データ追加による予測の安定化を体感する。